<a href="https://colab.research.google.com/github/Sanataniitb/autoimmune-disease/blob/main/notebooks/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoimmune disease classification: quickstart

This notebook runs the leak-free pipeline from the `autoimmune_ml` package on one disease cohort:

1. load a genes × samples expression table and label samples by name prefix,
2. cross-validate several classifiers with gene selection and scaling fitted **inside** each training fold,
3. report ROC AUC, balanced accuracy, sensitivity and specificity, plus which genes are selected consistently.

It runs on a synthetic dataset by default, so it works without access to the cohort files.

## 1. Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/Sanataniitb/autoimmune-disease.git
    %cd autoimmune-disease
    !pip install -q -e .
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the repository root so configs/ and data/ resolve

## 2. Choose the data

Leave `CONFIG` as the synthetic demo, or point it at a disease config (`configs/sle.yaml`, `configs/t1d.yaml`, ...) after placing the matching file in `data/` (see `data/README.md`). On Colab you can instead mount Google Drive and set `DATA_PATH`.

In [ ]:
from pathlib import Path
from autoimmune_ml import DiseaseConfig, make_synthetic

CONFIG = "configs/synthetic_demo.yaml"
DATA_PATH = None  # e.g. "/content/drive/MyDrive/.../GSE116006_SLE_N ml input.csv"

# from google.colab import drive; drive.mount("/content/drive")

cfg = DiseaseConfig.from_yaml(CONFIG)
if DATA_PATH:
    cfg.data_path = Path(DATA_PATH)
if cfg.name == "synthetic_demo" and not cfg.data_path.exists():
    cfg.data_path.parent.mkdir(exist_ok=True)
    make_synthetic().to_csv(cfg.data_path, index=False)
cfg

In [ ]:
from autoimmune_ml import load_expression
from autoimmune_ml.data import sample_groups

X, y = load_expression(cfg.data_path, cfg.positive_prefix, cfg.gene_column)
groups = sample_groups(X.index, cfg.group_regex)
print(X.shape, "disease:", y.sum(), "control:", (1 - y).sum())
X.iloc[:5, :8]

## 3. Cross-validate the models

Repeated stratified 5-fold CV. Increase `n_repeats` for tighter estimates; set `tune=True` for nested hyper-parameter search (slower).

In [ ]:
import pandas as pd
from autoimmune_ml import cross_validate_model
from autoimmune_ml.models import model_label

results = [
    cross_validate_model(
        X, y, m,
        importance_threshold=cfg.importance_threshold,
        always_include=cfg.always_include,
        log_transform=cfg.log_transform,
        groups=groups,
        n_splits=5, n_repeats=3,
    )
    for m in cfg.models
]
summary = pd.DataFrame([r.summary() for r in results])
summary.index = [model_label(r.model) for r in results]
summary[[c for c in summary.columns if c.endswith("_mean")]].round(3).sort_values("roc_auc_mean", ascending=False)

## 4. ROC curves and stable genes

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(5, 5))
for r in results:
    fpr, tpr, _ = roc_curve(y, r.oof_proba[0])
    ax.plot(fpr, tpr, label=f"{model_label(r.model)} (AUC {r.summary()['roc_auc_mean']:.2f})")
ax.plot([0, 1], [0, 1], "--", c="grey")
ax.set(xlabel="False positive rate", ylabel="True positive rate", title=f"{cfg.display_name}: out-of-fold ROC")
ax.legend(loc="lower right");

In [ ]:
best = max(results, key=lambda r: r.summary()["roc_auc_mean"])
print("Best model:", model_label(best.model))
best.feature_frequency().head(20).rename("fraction of folds selected").to_frame()

## 5. Command line equivalent

```bash
autoimmune-ml run configs/sle.yaml --output results/
```

writes `summary.csv`, per-fold metrics, ROC and confusion-matrix plots and `feature_stability.csv` to `results/<disease>/`.